# **DE PROJECT: REDDIT IPHONE VOC MEDALLION LAKEHOUSE & SERVING PIPELINE (CO5173)**

Hệ thống Kỹ thuật Dữ liệu (Data Engineering Pipeline) phân tích phản hồi người dùng iPhone từ dữ liệu Reddit theo kiến trúc **Medallion Lakehouse (Bronze $\rightarrow$ Silver $\rightarrow$ Gold $\rightarrow$ Serving)** tích hợp trực tiếp vào cấu trúc thư mục GitHub **`reddit-iphone-feedback-pipeline-main`**.

### **Luồng Kiến trúc Chính & Công nghệ Thay thế trong `PipelineExecutionConfig` (`config/pipeline_config.py`):**
- **Tầng Bronze (`ingestion/historical/extract/bronze_zst_ingestor.py` & `profiling/data_profiler.py`)**: Giải nén luồng `.zst` ($45,74\text{ triệu}$ bản ghi), lọc miền `r/iphone` ($4.205$ bài đăng), phân tích Data Profiling và lưu trữ cột nén **Parquet Snappy** kèm kiểm định toàn vẹn **SHA-256**.
- **Tầng Silver (`processing/batch/enrichment/silver_enrichment.py` & `nlp/`)**: Khử trùng lặp theo `id`, khôi phục $400$ bài đăng từ `crosspost_parent_list`, trích xuất thực thể `iphone_model` ($34,73\%$), `ios_version` ($7,93\%$), phân loại lỗi và gán nhãn cảm xúc bằng **PySpark Vectorized Pandas UDF (Transformer RoBERTa)** (kèm nhánh thay thế `alternatives/`: Lexicon UDF, Hadoop MapReduce RDD, DuckDB Vectorized).
- **Tầng Gold (`processing/batch/silver_to_gold/gold_datamarts.py` & `processing/streaming/`)**: Sử dụng **`batch_gold_engine = 'spark_sql'`** (Catalyst Vectorized) kết hợp **`stream_engine = 'apache_flink'`** (Stateful Sliding Window cho BR03) để xuất 8 bảng Gold Data Marts (BR01 $\rightarrow$ BR08) và đăng ký vào **Hive Lakehouse Catalog**.
- **Tầng Serving (`serving/database/postgres_loader.py` & `schema_ddl.sql`)**: Sử dụng **`serving_backend = 'postgres'`** với mô hình **`schema_paradigm = 'galaxy'`** (Fact Constellation chia sẻ Conformed Star Dimensions giữa 3 bảng Fact) đảm bảo độ trễ truy vấn $P_{99} \approx 5,48\text{ ms}$ và bộ nhớ RAM tối thiểu ($\approx 39\text{ KB}$).
- **Thực nghiệm Đối chuẩn & Đánh giá (`tests/pipeline_benchmark.py` - Đặt ở Mục 7 cuối cùng)**: Đối sánh thực nghiệm về **Độ trễ (Latency)**, **Dung lượng lưu trữ (Storage KB)** và **Bộ nhớ RAM đỉnh (Peak Memory KB)** giữa các công nghệ và mô hình Schema thay thế.


## **1. Kết nối Google Drive & Cài đặt Thư viện Môi trường**


In [ ]:
from google.colab import drive

# Kết nối với Google Drive để lưu trữ dữ liệu xuyên suốt các phiên làm việc
drive.mount("/content/drive")


In [ ]:
# Cài đặt aria2 và các thư viện xử lý dữ liệu, PySpark, NLP, DuckDB và Database
!apt-get update -qq
!apt-get install -y aria2
!pip install -q zstandard pandas pyarrow pyspark transformers torch sqlalchemy psycopg2-binary duckdb kafka-python tqdm


## **2. Giải nén Gói Mã nguồn GitHub (`reddit-iphone-feedback-pipeline-main.zip`) & Khởi tạo Cấu hình Hệ thống Chính**


In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path

EXTRACT_DIR = Path("/content/github_repo_workspace")
CANDIDATE_ZIPS = [
    Path("/content/reddit-iphone-feedback-pipeline-main.zip"),
    Path("/content/reddit-iphone-feedback-pipeline-updated.zip"),
    Path("/content/drive/MyDrive/DE/reddit-iphone-feedback-pipeline-main.zip"),
    Path("/content/drive/MyDrive/DE/reddit-iphone-feedback-pipeline-updated.zip"),
    Path("/content/DE_project.zip"),
    Path("/content/drive/MyDrive/DE/DE_project.zip")
]

zip_path = next((p for p in CANDIDATE_ZIPS if p.exists()), None)
if zip_path is None:
    from google.colab import files
    print("Vui lòng tải lên tệp nén GitHub (.zip): reddit-iphone-feedback-pipeline-main.zip")
    uploaded = files.upload()
    zip_path = Path("/content") / list(uploaded.keys())[0]

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

if (EXTRACT_DIR / "config").exists() and (EXTRACT_DIR / "processing").exists():
    REPO_ROOT = EXTRACT_DIR
elif (EXTRACT_DIR / "reddit-iphone-feedback-pipeline-main" / "config").exists():
    REPO_ROOT = EXTRACT_DIR / "reddit-iphone-feedback-pipeline-main"
else:
    subdirs = [d for d in EXTRACT_DIR.iterdir() if d.is_dir() and (d / "config").exists()]
    REPO_ROOT = subdirs[0] if subdirs else EXTRACT_DIR

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Xóa cache module cũ nếu chạy lại trong cùng phiên Colab
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("config", "ingestion", "nlp", "processing", "serving", "tests", "scripts")):
        del sys.modules[mod_name]

# Đóng gói lại đúng cấu trúc phẳng từ REPO_ROOT để phân phối cho PySpark Workers
SPARK_PACKAGE_ZIP = "/content/github_repo_spark_dist.zip"
if os.path.exists(SPARK_PACKAGE_ZIP):
    os.remove(SPARK_PACKAGE_ZIP)
shutil.make_archive("/content/github_repo_spark_dist", "zip", str(REPO_ROOT))

from config.pipeline_config import (
    StorageConfig,
    DatabaseConfig,
    DomainTaxonomyConfig,
    PipelineExecutionConfig
)
from ingestion.historical.extract.bronze_zst_ingestor import (
    stream_zst_json,
    extract_subreddit,
    preview_jsonl,
    serialize_nested,
    sha256_file,
    build_bronze_parquet
)
from ingestion.historical.profiling.data_profiler import profile_filtered_posts
from processing.batch.bronze_to_silver.spark_cleaner import build_spark_session
from processing.batch.enrichment.silver_enrichment import (
    run_silver_pipeline,
    benchmark_silver_enrichment_engines
)
from processing.batch.silver_to_gold.gold_datamarts import build_gold_datamarts
from processing.batch.silver_to_gold.alternatives.hadoop_mapreduce_gold import (
    run_mapreduce_word_and_issue_count
)
from processing.streaming.alternatives.apache_flink_engine import (
    benchmark_streaming_paradigms
)
from serving.database.alternatives.hive_metastore_catalog import (
    register_hive_metastore_tables
)
from serving.database.postgres_loader import UnifiedServingPipelineLoader
from tests.pipeline_benchmark import (
    run_schema_paradigm_benchmark,
    run_serving_latency_benchmark,
    select_optimal_pipeline_configuration,
    validate_medallion_quality_gates
)

# Khởi tạo Cấu hình Hệ thống Chính (Optimal Production Pipeline Config)
storage_cfg = StorageConfig()
storage_cfg.ensure_local_dirs()
taxonomy_cfg = DomainTaxonomyConfig()
production_exec_cfg = PipelineExecutionConfig()

print("Extracted GitHub Repo Root  :", REPO_ROOT)
print("Production Execution Config :", production_exec_cfg)
print("Bronze Parquet Path         :", storage_cfg.bronze_posts_parquet)
print("Silver Parquet Path         :", storage_cfg.silver_posts_path)
print("Gold Base Path              :", storage_cfg.gold_base_path)


## **3. Tầng Bronze: Thu nạp Dữ liệu, Lọc `r/iphone`, Profiling & Xuất Bronze Parquet**


In [ ]:
if not Path(storage_cfg.bronze_posts_parquet).exists() and not storage_cfg.posts_filtered_jsonl.exists():
    from google.colab import files
    uploaded = files.upload()
    uploaded_files = list(uploaded.keys())
    TORRENT_FILE = Path("/content") / uploaded_files[0]
    !aria2c -S "$TORRENT_FILE"
    !aria2c --select-file=2 --seed-time=0 --file-allocation=none "$TORRENT_FILE" -d /content/reddit_2026_07
    submission_files = list(storage_cfg.raw_local_dir.rglob("RS_2026-07.zst"))
    POSTS_RAW = submission_files[0]
    extract_summary = extract_subreddit(
        input_path=POSTS_RAW,
        output_path=storage_cfg.posts_filtered_jsonl,
        subreddit="iphone"
    )
else:
    print("Đã tìm thấy dữ liệu r/iphone đã lọc trên Google Drive, bỏ qua bước tải torrent.")


In [ ]:
if storage_cfg.posts_filtered_jsonl.exists():
    profile_results = profile_filtered_posts(
        jsonl_path=storage_cfg.posts_filtered_jsonl,
        profile_dir=storage_cfg.profile_dir
    )
    print("Overview:", profile_results["overview"])
    print("Degradation stats:", profile_results["degradation_stats"])
    display(profile_results["numeric_profile"])


In [ ]:
if storage_cfg.posts_filtered_jsonl.exists():
    bronze_report = build_bronze_parquet(
        jsonl_path=storage_cfg.posts_filtered_jsonl,
        parquet_path=storage_cfg.bronze_posts_parquet
    )
    print("===== BRONZE PARQUET REPORT =====")
    for k, v in bronze_report.items():
        if k != "nested_columns":
            print(f"{k}: {v}")


## **4. Tầng Silver: Làm sạch Văn bản, Khôi phục Crosspost, Trích xuất Thực thể & Phân tích Cảm xúc (PySpark)**


In [ ]:
from processing.batch.bronze_to_silver.spark_cleaner import build_spark_session
from processing.batch.enrichment.silver_enrichment import (
    run_silver_pipeline,
    benchmark_silver_enrichment_engines
)

# Ở Mục 4: Truyền trực tiếp production_exec_cfg vào run_silver_pipeline
spark = build_spark_session("DE_Project_VoC_Medallion", storage_cfg)
spark.sparkContext.addPyFile(SPARK_PACKAGE_ZIP)

silver_sdf = run_silver_pipeline(spark, storage_cfg, taxonomy_cfg, exec_cfg=production_exec_cfg)

print("Silver row count:", silver_sdf.count())
silver_sdf.filter("iphone_model != 'Other / Unspecified' OR ios_version != 'iOS Unspecified'").select(
    "post_id", "iphone_model", "ios_version", "feature_category",
    "issue_category", "sentiment", "sentiment_score", "engagement_score", "crosspost_recovered"
).show(15, truncate=False)


## **5. Tầng Gold: Thực thi Luồng Chính (`production_exec_cfg`) Xuất 8 Tập Dữ liệu Nghiệp vụ (BR01 $\rightarrow$ BR08)**


In [ ]:
gold_marts = build_gold_datamarts(spark, storage_cfg, exec_cfg=production_exec_cfg)
registered_tables = register_hive_metastore_tables(spark, storage_cfg.gold_base_path)
print("Registered Hive Lakehouse Catalog Tables:", registered_tables)

for mart_name, mart_sdf in gold_marts.items():
    print(f"=== GOLD MART: {mart_name} (Rows: {mart_sdf.count()}) ===")
    mart_sdf.show(5, truncate=False)


## **6. Tầng Serving: Đồng bộ Lũy đẳng (Idempotent Upsert) vào Cơ sở Dữ liệu Phục vụ & Truy vấn Kiểm chứng**
Mặc định `use_sqlite_fallback=True` giúp lưu trữ và truy vấn kiểm chứng ngay tại `/content/drive/MyDrive/DE/serving_voc_star.db`. Khi kết nối tới **Cloud PostgreSQL (Supabase / Neon)**, chỉ cần đổi `use_sqlite_fallback=False` và điền thông tin `host`, `user`, `password`.


In [ ]:
db_cfg = DatabaseConfig(
    host="localhost",
    port=5432,
    database="voc_analytics",
    user="postgres",
    password="postgres",
    use_sqlite_fallback=True  # Đổi thành False khi nạp lên Cloud PostgreSQL (Supabase/Neon)
)

serving_loader = UnifiedServingPipelineLoader(storage_cfg, db_cfg, exec_cfg=production_exec_cfg)
serving_loader.initialize_schema()
sync_stats = serving_loader.sync_dimensions_and_facts()
print("===== SERVING LAYER SYNCHRONIZATION COMPLETED =====")
for k, v in sync_stats.items():
    print(f"{k}: {v}")

serving_result_df = serving_loader.query_serving_mart()
print(f"\n===== ANALYTICAL QUERY RESULT FROM SERVING LAYER ({production_exec_cfg.schema_paradigm.upper()} SCHEMA) =====")
display(serving_result_df.head(15))


## **7. Thực nghiệm Đối chuẩn & Đánh giá Toàn diện (Benchmarking & Architecture Justification)**
Phần thực nghiệm dưới đây đặt ở cuối cùng nhằm cung cấp toàn bộ bảng số liệu định lượng (**Độ trễ $P_{50}/P_{95}/P_{99}$**, **Dung lượng lưu trữ KB**, **Bộ nhớ RAM đỉnh KB**) chứng minh tại sao cấu hình `production_exec_cfg` được chọn làm luồng xử lý chính của hệ thống.


In [ ]:
import pandas as pd

# 7.1. Kiểm định chất lượng dữ liệu (Data Correctness) và bộ nhớ tầng Silver
silver_pdf = pd.read_parquet(storage_cfg.silver_posts_path)
quality_report = validate_medallion_quality_gates(silver_pdf)
print("===== 1A. DATA CORRECTNESS & MEMORY FOOTPRINT =====")
for k, v in quality_report.items():
    print(f"{k}: {v}")

# 7.2. Đối chuẩn 4 Công nghệ Tầng Silver (Bronze -> Silver Enrichment & NLP Engines)
silver_bench_df = benchmark_silver_enrichment_engines(
    spark=spark,
    storage_cfg=storage_cfg,
    taxonomy_cfg=taxonomy_cfg,
    sample_limit=500
)
print("\n===== 1B. SILVER ENRICHMENT & NLP ENGINES BENCHMARK (BRONZE -> SILVER) =====")
display(silver_bench_df)

# 7.3. Đối chuẩn 4 Kiến trúc Schema (Flat OBT vs. Star vs. Snowflake vs. Galaxy)
schema_bench_df = run_schema_paradigm_benchmark(storage_cfg.silver_posts_path, iterations=40)
print("\n===== 2. SCHEMA PARADIGM BENCHMARK (LATENCY + STORAGE KB + PEAK MEMORY KB) =====")
display(schema_bench_df)

# 7.4. Đối chuẩn Batch Processing Engines (Hadoop MapReduce RDD vs. Spark SQL vs. DuckDB)
batch_bench_df = run_mapreduce_word_and_issue_count(spark, storage_cfg.silver_posts_path)
print("\n===== 3A. BATCH PROCESSING ENGINES (LATENCY + DRIVER PEAK MEMORY KB) =====")
display(batch_bench_df)

# 7.5. Đối chuẩn Streaming Engines (Flink vs. Kafka Streams vs. Spark Structured Streaming)
stream_bench_df = benchmark_streaming_paradigms(storage_cfg.silver_posts_path)
print("\n===== 3B. STREAMING ENGINES (LATENCY + THROUGHPUT + PEAK STATE MEMORY KB) =====")
display(stream_bench_df)

# 7.6. Đối chuẩn Công nghệ Lưu trữ & Phục vụ (JSONL vs. Parquet vs. DuckDB BigLake vs. Indexed RDBMS)
mgmt_bench_df = run_serving_latency_benchmark(
    jsonl_path=storage_cfg.posts_filtered_jsonl,
    silver_parquet_path=storage_cfg.silver_posts_path,
    iterations=30
)
print("\n===== 4. DATA MANAGEMENT & SERVING BENCHMARK (LATENCY + STORAGE KB + PEAK MEMORY KB) =====")
display(mgmt_bench_df)

# 7.7. Ma trận tổng hợp quyết định kiến trúc toàn trình (Bao gồm cả Tầng Silver -> Gold -> Serving)
_, decision_matrix_df = select_optimal_pipeline_configuration(
    schema_bench_df=schema_bench_df,
    batch_bench_df=batch_bench_df,
    stream_bench_df=stream_bench_df,
    mgmt_bench_df=mgmt_bench_df,
    silver_bench_df=silver_bench_df
)
print("\n===== 5. MULTI-CRITERIA ARCHITECTURE DECISION SUMMARY (ALL 5 STAGES) =====")
display(decision_matrix_df)